In [15]:
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

In [16]:
# Read grayscale braille image
# img = cv2.imread("./images/raw.png", cv2.IMREAD_GRAYSCALE)
img = cv2.imread("./images/braille_image_01.jpeg", cv2.IMREAD_GRAYSCALE)

assert img is not None, "Image not found"

H, W = img.shape
# print("Image shape:", img.shape)

In [17]:
img_norm = img.astype(np.float32) / 255.0
img_3ch = np.stack([img_norm]*3, axis=0)   # (3, H, W)
img_tensor = torch.from_numpy(img_3ch).unsqueeze(0)  # (1, 3, H, W)

In [18]:
def gaussian_heatmap_2d(H, W, cx, cy, sigma_x, sigma_y, A=1.0):
    y = np.arange(H)
    x = np.arange(W)
    xx, yy = np.meshgrid(x, y)

    heatmap = A * np.exp(
        -(((xx - cx) ** 2) / (2 * sigma_x ** 2) +
          ((yy - cy) ** 2) / (2 * sigma_y ** 2))
    )
    return heatmap

In [ ]:
# Example braille dot centers (x, y)
dot_centers = [
    (120, 80),
    (160, 80),
    (120, 120),
    (160, 120)
]
gt_heatmap = np.zeros((H, W), dtype=np.float32)

for (cx, cy) in dot_centers:
    g = gaussian_heatmap_2d(H, W, cx, cy, sigma_x=3, sigma_y=3)
    gt_heatmap = np.maximum(gt_heatmap, g)

heatmap_vis = (gt_heatmap * 255).astype(np.uint8)
heatmap_color = cv2.applyColorMap(heatmap_vis, cv2.COLORMAP_JET)

cv2.imshow("Gaussian Heatmap (GT)", heatmap_color)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [21]:
class ResNet50Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights="IMAGENET1K_V1")

        self.stage1 = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool
        )
        self.stage2 = resnet.layer1
        self.stage3 = resnet.layer2
        self.stage4 = resnet.layer3
        self.stage5 = resnet.layer4

    def forward(self, x):
        f1 = self.stage1(x)
        f2 = self.stage2(f1)
        f3 = self.stage3(f2)
        f4 = self.stage4(f3)
        f5 = self.stage5(f4)
        return f1, f2, f3, f4, f5

In [22]:
class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, out_ch, 2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.up(x)

        # 🔧 FIX: force spatial alignment
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

In [23]:
class BddNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = ResNet50Encoder()

        self.dec4 = DecoderBlock(2048, 1024, 512)
        self.dec3 = DecoderBlock(512, 512, 256)
        self.dec2 = DecoderBlock(256, 256, 128)
        self.dec1 = DecoderBlock(128, 64, 64)

        self.out_conv = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        f1, f2, f3, f4, f5 = self.encoder(x)
        x = self.dec4(f5, f4)
        x = self.dec3(x, f3)
        x = self.dec2(x, f2)
        x = self.dec1(x, f1)
        return torch.sigmoid(self.out_conv(x))

In [24]:
model = BddNet()
model.eval()

with torch.no_grad():
    pred = model(img_tensor)

pred_heatmap = pred.squeeze().cpu().numpy()

In [25]:
pred_vis = (pred_heatmap * 255).astype(np.uint8)
pred_color = cv2.applyColorMap(pred_vis, cv2.COLORMAP_JET)

cv2.imshow("Predicted Heatmap", pred_color)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [26]:
img_color = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)

pred_color_resized = cv2.resize(
    pred_color,
    (img_color.shape[1], img_color.shape[0]),
    interpolation=cv2.INTER_LINEAR
)

overlay = cv2.addWeighted(
    img_color, 0.6,
    pred_color_resized, 0.4,
    0
)

cv2.imshow("Overlay", overlay)
cv2.waitKey(0)
cv2.destroyAllWindows()

In [27]:
# Get predicted spatial size
_, _, H_pred, W_pred = pred.shape

# Resize GT heatmap to match prediction
gt_resized = cv2.resize(
    gt_heatmap,
    (W_pred, H_pred),
    interpolation=cv2.INTER_LINEAR
)

gt_tensor = torch.from_numpy(gt_resized).unsqueeze(0).unsqueeze(0).float()


In [28]:
criterion = nn.MSELoss()
loss = criterion(pred, gt_tensor)

print("L2 Loss:", loss.item())

L2 Loss: 0.2606241703033447


In [8]:
import os
import cv2
import torch
import json
import numpy as np
import math
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
def preprocess_image(img, size=(512, 512)):
    img = cv2.resize(img, size, interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    return img

def visualize(img, gt, pred=None):
    plt.figure(figsize=(12,4))
    plt.subplot(1,3,1); plt.imshow(img, cmap='gray'); plt.title("Input")
    plt.subplot(1,3,2); plt.imshow(gt, cmap='hot'); plt.title("Ground Truth")
    if pred is not None:
        plt.subplot(1,3,3); plt.imshow(pred, cmap='hot'); plt.title("Prediction")
    plt.show()
    # cv2.imshow("pred", pred)
    # cv2.waitKey(0)
    # cv2.destroyAllWindows()

In [10]:
# ========== MODEL.PY ==========

# ---------------------------
# Upsampling Block (U-Net)
# ---------------------------
class UpBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up = nn.ConvTranspose2d(
            in_ch, out_ch, kernel_size=2, stride=2
        )
        self.conv = nn.Sequential(
            nn.Conv2d(out_ch + skip_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


# ---------------------------
# Braille Dot Detection Network
# ---------------------------
class BddNet(nn.Module):
    def __init__(self, input_size=512):
        super().__init__()
        self.input_size = input_size

        # -------- ResNet-50 Encoder --------
        resnet = models.resnet50(
            weights=models.ResNet50_Weights.IMAGENET1K_V1
        )

        # ---- Grayscale input (1 channel) ----
        old_conv = resnet.conv1
        resnet.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )
        resnet.conv1.weight.data = old_conv.weight.data.mean(dim=1, keepdim=True)

        self.enc1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)  # 64
        self.pool = resnet.maxpool                                      # ↓
        self.enc2 = resnet.layer1                                       # 256
        self.enc3 = resnet.layer2                                       # 512
        self.enc4 = resnet.layer3                                       # 1024
        self.enc5 = resnet.layer4                                       # 2048

        # -------- Bottleneck --------
        self.center = nn.Sequential(
            nn.Conv2d(2048, 1024, kernel_size=3, padding=1),
            nn.BatchNorm2d(1024),
            nn.ReLU(inplace=True)
        )

        # -------- Decoder (FIXED) --------
        self.up4 = UpBlock(1024, 1024, 512)
        self.up3 = UpBlock(512, 512, 256)
        self.up2 = UpBlock(256, 256, 128)
        self.up1 = UpBlock(128, 64, 64)

        # -------- Final Heatmap --------
        self.final = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        # -------- Encoder --------
        e1 = self.enc1(x)             # [B, 64,  H/2,  W/2]
        e2 = self.enc2(self.pool(e1)) # [B, 256, H/4,  W/4]
        e3 = self.enc3(e2)            # [B, 512, H/8,  W/8]
        e4 = self.enc4(e3)            # [B, 1024,H/16, W/16]
        e5 = self.enc5(e4)            # [B, 2048,H/32, W/32]

        # -------- Center --------
        c = self.center(e5)           # [B, 1024,H/32, W/32]

        # -------- Decoder --------
        x = self.up4(c, e4)           # [B, 512, H/16, W/16]
        x = self.up3(x, e3)           # [B, 256, H/8,  W/8]
        x = self.up2(x, e2)           # [B, 128, H/4,  W/4]
        x = self.up1(x, e1)           # [B, 64,  H/2,  W/2]

        x = self.final(x)             # [B, 1,   H/2,  W/2]

        # -------- Resize to GT size --------
        x = F.interpolate(
            x,
            size=(self.input_size, self.input_size),
            mode="bilinear",
            align_corners=False
        )

        return torch.sigmoid(x)

In [11]:
# ========== TEST.PY ==========

def infer_single(image_path, checkpoint, out_path=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = BddNet().to(device)
    model.load_state_dict(torch.load(checkpoint, map_location=device))
    model.eval()

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not load image: {image_path}")
    
    inp = preprocess_image(img)
    inp = torch.from_numpy(inp).unsqueeze(0).unsqueeze(0).float().to(device)
    with torch.no_grad():
        pred = model(inp)[0,0].cpu().numpy()
    visualize(img, np.zeros_like(img), pred)
    if out_path:
        cv2.imwrite(out_path, (pred*255).astype(np.uint8))
    return pred

In [12]:
# Example inference (uncomment and modify paths as needed)
image_path = "DSBI-master\data\Ordinary Printed Document\OPD+1+recto.jpg"   # Path to input image
checkpoint = "DSBI-master\src\checkpoints\checkpoint_epoch_6.pt"  # Path to trained model checkpoint
output_path = "output_prediction.jpg"   # Path to save prediction (optional)

pred = infer_single(image_path, checkpoint, output_path)

C:\Users\rohan\AppData\Local\Temp\ipykernel_4184\4188511750.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint, map_location=dev